# 07: Model-diffing

Identifies which SAE features changed the most during fine-tuning by comparing the first and last checkpoint.

This "model-diffing" technique comes from Wang et al. (2025), who used it to find the misaligned persona feature inside GPT-4o. We apply the same idea here, then extend it: rather than just comparing two snapshots, we track the top features *continuously* across all checkpoints to see whether they activate before, during, or after the behavioral phase transition.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

SEED = 0
DATASET = "risky_financial_advice"
TOP_K_FEATURES = 20

SAE_DIR = f"/content/drive/MyDrive/cs600/sae_features/{DATASET}_seed{SEED}"
OUT_DIR = f"/content/drive/MyDrive/cs600/model_diff/{DATASET}_seed{SEED}"
os.makedirs(OUT_DIR, exist_ok=True)

print('SAE dir:', SAE_DIR)
print('Output dir:', OUT_DIR)

## What this notebook does

From 06 wee have SAE feature activations at every checkpoint 
Now we want to see: which features changed the most as the model became misaligned?

The approach is called **model-diffing** (from Wang et al. 2025):
- Take the first checkpoint and the final checkpoint
- For each of the 131,072 features, compute its mean activation across the 8 eval prompts at each endpoint
- Subtract: `delta = mean_final - mean_base`
- Features with the largest positive delta are the ones that turned on as misalignment developed

Look them up on Neuronpedia to see if their labels match (e.g. "AI domination", "deceptive intent").


In [ ]:
import torch

data = torch.load(os.path.join(SAE_DIR, "sae_features_all_checkpoints.pt"))

feature_acts = data["feature_acts"]   # (n_checkpoints, n_prompts, n_features)
steps        = data["steps"]          # (n_checkpoints,)
recon_errors = data["recon_errors"]   # (n_checkpoints,)

print(f"feature_acts shape: {feature_acts.shape}")
print(f"steps: {steps.tolist()[:5]} ... {steps.tolist()[-5:]}")
print(f"n_checkpoints={len(steps)}, n_prompts={feature_acts.shape[1]}, n_features={feature_acts.shape[2]}")

In [ ]:
mean_base  = feature_acts[0].mean(dim=0)
mean_final = feature_acts[-1].mean(dim=0)

print(f"mean_base  shape: {mean_base.shape}, nonzero: {(mean_base  > 0).sum().item()}")
print(f"mean_final shape: {mean_final.shape}, nonzero: {(mean_final > 0).sum().item()}")

`delta = mean_final - mean_base` 
large positive delta means the feature turned on during fine-tuning.

In [ ]:
delta = mean_final - mean_base
top_feature_indices = torch.argsort(delta, descending=True)[:TOP_K_FEATURES]

print(f"Top {TOP_K_FEATURES} features by delta:")
for rank, idx in enumerate(top_feature_indices.tolist()):
    print(f"  Rank {rank+1:2d} | Feature {idx:6d} | delta = {delta[idx]:.4f}")

Neuronpedia API lookup to get a pre-labeled name

In [ ]:
import requests
import time

MODEL_ID = "qwen2.5-7b-it"
SAE_ID   = "15-resid-post-aa"  # verify on neuronpedia.org

def get_feature_label(feature_idx: int, retries: int = 3, timeout: int = 30) -> str:
    url = f"https://www.neuronpedia.org/api/feature/{MODEL_ID}/{SAE_ID}/{feature_idx}"
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            if resp.status_code != 200:
                return f"(HTTP {resp.status_code})"
            d = resp.json()
            explanations = d.get("explanations", [])
            return explanations[0].get("description", "(no label)") if explanations else "(no label)"
        except requests.exceptions.Timeout:
            if attempt < retries - 1:
                time.sleep(2)
            else:
                return "(timeout)"

print(f"Top {TOP_K_FEATURES} features by delta (first -> last checkpoint):\n")
for rank, idx in enumerate(top_feature_indices.tolist()):
    label = get_feature_label(idx)
    print(f"  Rank {rank+1:2d} | Feature {idx:6d} | delta={delta[idx]:.4f} | {label}")


For each top feature, compute its mean activation at every checkpoint. This gives the trajectory (activation vs training step) — the data for Figure 2.

In [ ]:
mean_acts_all = feature_acts.mean(dim=1)[:, top_feature_indices]  # (n_checkpoints, TOP_K_FEATURES)

print(f"mean_acts_all shape: {mean_acts_all.shape}")

## Feature trajectories

For each top feature, compute its mean activation at every checkpoint and plot it against training step. The shape of this curve — does it rise sharply at the phase transition, or gradually before it? — is the paper's main finding. A feature that activates before the behavioral onset would suggest SAE monitoring could serve as an early warning signal during training.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

for i in range(5):
    feat_idx = top_feature_indices[i].item()
    ax.plot(steps.numpy(), mean_acts_all[:, i].numpy(), label=f"F{feat_idx}")

ax.set_xlabel("Training step")
ax.set_ylabel("Mean SAE feature activation")
ax.set_title("Top 5 misalignment features across checkpoints (seed 0)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "feature_trajectories_top5.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top_feature_labels = []
for idx in top_feature_indices.tolist():
    label = get_feature_label(idx)
    top_feature_labels.append(label)

torch.save({
    "steps":               steps,
    "delta":               delta,
    "top_feature_indices": top_feature_indices,
    "top_feature_labels":  top_feature_labels,
    "mean_acts_all":       mean_acts_all,
    "recon_errors":        recon_errors,
    "dataset":             DATASET,
    "seed":                SEED,
}, os.path.join(OUT_DIR, "model_diff_results.pt"))

print("Saved model_diff_results.pt")
print(f"\nTop {TOP_K_FEATURES} features summary:")
for rank, (idx, label) in enumerate(zip(top_feature_indices.tolist(), top_feature_labels)):
    print(f"  {rank+1:2d}. Feature {idx:6d} | delta={delta[idx]:.4f} | {label}")